In [6]:
import os
import stat
import posixpath
from pathlib import Path
from getpass import getpass
import paramiko
from dotenv import load_dotenv

In [7]:
load_dotenv()  # Carrega variáveis de ambiente do arquivo .env

True

In [8]:
HOST         = os.getenv("HOST")
PORT         = int(os.getenv("PORT"))
USERNAME     = os.getenv("USERNAME", "bruno")
KEY_FILE     = os.path.expanduser(os.getenv("PASSKEY_PATH"))
#PASSWORD     = None  

LOCAL_FILE   = Path(os.getenv("LOCAL_FILE_REPORT"))

# Caminho REMOTO final do HTML (onde o Nginx aponta)
REMOTE_PATH  = os.getenv(
    "REMOTE_PATH_REPORTS"
)

In [9]:
def _load_pkey(path: str):
    if not path or not Path(path).expanduser().exists():
        return None
    p = str(Path(path).expanduser())
    try:
        return paramiko.Ed25519Key.from_private_key_file(p)
    except paramiko.ssh_exception.PasswordRequiredException:
        pw = getpass(f"Passphrase para {p}: ")
        return paramiko.Ed25519Key.from_private_key_file(p, password=pw)
    except Exception:
        pass
    try:
        return paramiko.RSAKey.from_private_key_file(p)
    except paramiko.ssh_exception.PasswordRequiredException:
        pw = getpass(f"Passphrase para {p}: ")
        return paramiko.RSAKey.from_private_key_file(p, password=pw)

def _sftp_mkdirs(sftp: paramiko.SFTPClient, remote_dir: str):
    parts = remote_dir.strip("/").split("/")
    cur = ""
    for part in parts:
        cur = f"{cur}/{part}" if cur else f"/{part}"
        try:
            sftp.stat(cur)
        except FileNotFoundError:
            sftp.mkdir(cur, mode=0o755)

def _exists(sftp: paramiko.SFTPClient, path: str) -> bool:
    try:
        sftp.stat(path)
        return True
    except FileNotFoundError:
        return False

def main():
    if not LOCAL_FILE.exists():
        raise FileNotFoundError(f"Local não encontrado: {LOCAL_FILE.resolve()}")

    print(f"Upload ...")

    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    pkey = _load_pkey(KEY_FILE) if KEY_FILE else None
    agent = paramiko.Agent()
    agent_keys = agent.get_keys()

    kwargs = dict(hostname=HOST, port=PORT, username=USERNAME, timeout=20, look_for_keys=False, allow_agent=True)
    if pkey:
        kwargs["pkey"] = pkey
    elif agent_keys:
        pass
    # elif PASSWORD:
    #     kwargs["password"] = PASSWORD
    client.connect(**kwargs)

    sftp = client.open_sftp()

    # Normaliza e garante diretório
    remote_final = sftp.normalize(REMOTE_PATH)
    remote_dir   = posixpath.dirname(remote_final)
    _sftp_mkdirs(sftp, remote_dir)

    tmp_remote = remote_final + ".tmp"
    bak_remote = remote_final + ".bak"

    # 1) put para .tmp
    sftp.put(str(LOCAL_FILE), tmp_remote)
    sftp.chmod(tmp_remote, 0o644)

    # 2) se arquivo final existir, tenta mover para .bak (fallback: remove)
    if _exists(sftp, remote_final):
        try:
            sftp.rename(remote_final, bak_remote)
        except Exception:
            # se o servidor não permitir rename (ou já existir .bak), remova o final
            try:
                sftp.remove(remote_final)
            except Exception as e:
                sftp.remove(tmp_remote)  # rollback
                sftp.close(); client.close()
                raise RuntimeError(f"Não foi possível substituir {remote_final}: {e}")

    # 3) renomeia .tmp -> final
    try:
        sftp.rename(tmp_remote, remote_final)
    except Exception as e:
        # tentativa de rollback: se houver .bak, volta
        if _exists(sftp, bak_remote):
            try:
                sftp.rename(bak_remote, remote_final)
            except Exception:
                pass
        # limpa o tmp
        try:
            sftp.remove(tmp_remote)
        except Exception:
            pass
        sftp.close(); client.close()
        raise RuntimeError(f"Falha ao mover .tmp para destino: {e}")

    # 4) permissões finais
    try:
        sftp.chmod(remote_final, 0o644)
    except Exception:
        pass

    # 5) limpeza do .bak (opcional: comente se quiser manter backup)
    # if _exists(sftp, bak_remote):
    #     try:
    #         sftp.remove(bak_remote)
    #     except Exception:
    #         pass

    sftp.close()
    client.close()
    print("✅ Upload concluído.")


In [10]:
main()

Upload ...
✅ Upload concluído.
